# Data Load and Image Processing

## Data Load

In [ ]:
# ==============================================================================
# 0. GLOBAL CONFIG & STABILITY FIXES
# ==============================================================================
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"

import io
import math
import time
import requests
import json
import re
import gc
import torch
import pandas as pd
from PIL import Image, ImageDraw
from tqdm import tqdm


from PIL import ImageEnhance

base_folder = '/content/drive/MyDrive/Iran Israel War/extracted_map_layers'
DRIVE_BASE    = '/content/drive/MyDrive/Iran Israel War/extracted_map_layers'    # your project folder
# INPUT_CSV     = os.path.join(DRIVE_BASE, 'final_extracted_events.csv')
INPUT_CSV     = os.path.join(DRIVE_BASE, 'master_unified_campaign_log.csv')
os.makedirs(base_folder, exist_ok=True)
csv_path = INPUT_CSV
output_dir = os.path.join(base_folder, "impact_maps_final_no_label")
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path)
df.rename(columns={'final_calculated_radius_m':'max_crater_radius_m',"latitude":'lat_dec','longitude':'lon_dec'},inplace=True)

df_valid = df[(df['max_crater_radius_m'] > 0) ].copy()

In [ ]:
df.shape

(893, 14)

In [ ]:
df_valid.shape

(892, 14)

## Image Processing

### Helper Functions

In [ ]:

# ==============================================================================
# PHASE 1: DYNAMIC SATELLITE & ROAD MAPS (TWO SEPARATE IMAGES)
# ==============================================================================
print("--- PHASE 1: GENERATING DYNAMIC SATELLITE & ROAD MAPS ---")

def dms_to_decimal(dms_str):
    if pd.isna(dms_str): return float('nan')
    if isinstance(dms_str, (int, float)): return float(dms_str)
    dms_str = str(dms_str).strip().upper()
    match = re.search(r"(\d+)[°\s]+(\d+)[′'\s]+(?:(\d+(?:\.\d+)?)[″\"\s]+)?([NSEW])", dms_str)
    if match:
        degrees = float(match.group(1)); minutes = float(match.group(2))
        seconds = float(match.group(3)) if match.group(3) else 0.0
        direction = match.group(4)
        decimal = degrees + (minutes / 60.0) + (seconds / 3600.0)
        if direction in ['S', 'W']: decimal *= -1
        return decimal
    try: return float(dms_str)
    except ValueError: return float('nan')

def calculate_optimal_zoom(lat, max_radius_m, image_size=512):
    if max_radius_m <= 0: return 20
    target_px = (image_size / 2) * 0.90
    m_per_px_target = max_radius_m / target_px
    val = (156543.03392 * math.cos(math.radians(lat))) / m_per_px_target
    zoom = math.log2(val)
    return max(15, min(20, math.floor(zoom)))


def get_static_satellite_image(lat, lon, zoom, size=(512, 512)):
    lat_rad = math.radians(lat)
    n = 2.0 ** zoom
    x_point = ((lon + 180.0) / 360.0) * 256.0 * n
    y_point = (1.0 - math.log(math.tan(lat_rad) + (1 / math.cos(lat_rad))) / math.pi) / 2.0 * 256.0 * n
    tile_x, tile_y = int(x_point // 256), int(y_point // 256)
    url_template = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"
    full_image = Image.new('RGB', (256*3, 256*3))
    for i, dx in enumerate([-1, 0, 1]):
        for j, dy in enumerate([-1, 0, 1]):
            resp = requests.get(url_template.format(x=tile_x + dx, y=tile_y + dy, z=zoom))
            if resp.status_code == 200: full_image.paste(Image.open(io.BytesIO(resp.content)), (i*256, j*256))
    exact_px, exact_py = int(x_point - ((tile_x - 1) * 256)), int(y_point - ((tile_y - 1) * 256))
    left, top = exact_px - size[0] // 2, exact_py - size[1] // 2
    return full_image.crop((left, top, left + size[0], top + size[1]))
import math
import requests
import io
from PIL import Image

# def get_esri_satellite_image(lat, lon, zoom, size=(512, 512), min_zoom=0, min_valid_tiles=5):
#     """
#     Tries to fetch ESRI tiles. If current zoom fails, progressively decreases zoom.

#     Args:
#         lat, lon: coordinates
#         zoom: starting zoom level
#         size: output image size
#         min_zoom: lowest zoom allowed
#         min_valid_tiles: minimum tiles required to accept a zoom level

#     Returns:
#         PIL Image
#     """

#     url_template = "https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
#     headers = {"User-Agent": "Mozilla/5.0"}

#     while zoom >= min_zoom:
#         n = 2.0 ** zoom

#         x_exact = (lon + 180.0) / 360.0 * n
#         y_exact = (1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n

#         center_x_tile, center_y_tile = int(x_exact), int(y_exact)

#         canvas = Image.new('RGB', (256 * 3, 256 * 3))
#         valid_tiles = 0

#         for dx in [-1, 0, 1]:
#             for dy in [-1, 0, 1]:
#                 url = url_template.format(
#                     x=center_x_tile + dx,
#                     y=center_y_tile + dy,
#                     z=zoom
#                 )

#                 try:
#                     resp = requests.get(url, headers=headers, timeout=5)

#                     if resp.status_code == 200:
#                         tile = Image.open(io.BytesIO(resp.content)).convert("RGB")

#                         # Optional: detect blank tiles (very important)
#                         if tile.getbbox() is not None:
#                             canvas.paste(tile, ((dx + 1) * 256, (dy + 1) * 256))
#                             valid_tiles += 1

#                 except Exception:
#                     continue

#         # ✅ Accept this zoom if enough tiles are valid
#         if valid_tiles >= min_valid_tiles:
#             offset_x_px = int((x_exact - center_x_tile) * 256)
#             offset_y_px = int((y_exact - center_y_tile) * 256)

#             left = 256 + offset_x_px - size[0] // 2
#             top = 256 + offset_y_px - size[1] // 2

#             return canvas.crop((left, top, left + size[0], top + size[1]))

#         # ❌ Otherwise fallback to lower zoom
#         zoom -= 1
#         print(f"[INFO] Falling back to zoom {zoom}")

#     # 🚨 If everything fails
#     raise ValueError("Could not fetch valid tiles at any zoom level")
def get_static_roadmap_image(lat, lon, zoom, size=(512, 512)):
    n = 2.0 ** zoom
    x_exact, y_exact = (lon + 180.0) / 360.0 * n, (1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n
    center_x_tile, center_y_tile = int(x_exact), int(y_exact)
    url_template = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
    canvas = Image.new('RGB', (256 * 3, 256 * 3))
    headers = {"User-Agent": "Mozilla/5.0"}
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            resp = requests.get(url_template.format(x=center_x_tile + dx, y=center_y_tile + dy, z=zoom), headers=headers)
            if resp.status_code == 200: canvas.paste(Image.open(io.BytesIO(resp.content)).convert("RGB"), ((dx + 1) * 256, (dy + 1) * 256))
    offset_x_px, offset_y_px = int((x_exact - center_x_tile) * 256), int((y_exact - center_y_tile) * 256)
    left, top = 256 + offset_x_px - size[0] // 2, 256 + offset_y_px - size[1] // 2
    return canvas.crop((left, top, left + size[0], top + size[1]))

def draw_bda_rings(img, lat, r_red, zoom):
    img_rgba = img.convert("RGBA")
    overlay = Image.new("RGBA", img_rgba.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    cx, cy = img_rgba.width // 2, img_rgba.height // 2

    m_per_px = 156543.03392 * math.cos(math.radians(lat)) / (2 ** zoom)

    if r_red > 0:
        px_r = r_red / m_per_px
        # CHANGED: fill=None completely removes the center tint.
        # CHANGED: width=5 makes the boundary unmissable for weak AI.
        draw.ellipse(
            (cx-px_r, cy-px_r, cx+px_r, cy+px_r),
            fill=(255, 0, 0, 20),
            outline=(255, 0, 0, 255),
            width=5
        )

    # Optional: You might even want to remove the crosshairs if the AI is confusing them for buildings
    # draw.line((cx - 10, cy, cx + 10, cy), fill="white", width=2)
    # draw.line((cx, cy - 10, cx, cy + 10), fill="white", width=2)

    return Image.alpha_composite(img_rgba, overlay).convert("RGB")

--- PHASE 1: GENERATING DYNAMIC SATELLITE & ROAD MAPS ---


In [ ]:
import math
import requests
import io
import numpy as np
from PIL import Image

# ==============================
# 🔍 Strong tile validation
# ==============================
def is_valid_tile(tile):
    arr = np.array(tile)

    # 1. Very low variance → reject
    if arr.std() < 10:
        return False

    # 2. Low color diversity → reject
    unique_colors = len(np.unique(arr.reshape(-1, 3), axis=0))
    if unique_colors < 500:
        return False

    # 3. Dominant gray detection (KEY for "no data" tiles)
    mean_color = arr.mean(axis=(0, 1))
    if abs(mean_color[0] - mean_color[1]) < 5 and abs(mean_color[1] - mean_color[2]) < 5:
        if arr.std() < 25:
            return False

    # 4. Bright text on gray (detect "Map data not available")
    if (arr > 220).sum() > 2000 and arr.std() < 30:
        return False

    return True


# ==============================
# 🌍 Main function with fallback
# ==============================
def get_esri_satellite_image(
    lat,
    lon,
    zoom,
    size=(512, 512),
    min_zoom=0,
    min_valid_tiles=7,
    timeout=5,
    verbose=True
):
    url_template = "https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
    headers = {"User-Agent": "Mozilla/5.0"}

    while zoom >= min_zoom:
        n = 2.0 ** zoom

        # Convert lat/lon → tile coords
        x_exact = (lon + 180.0) / 360.0 * n
        y_exact = (1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n

        center_x_tile, center_y_tile = int(x_exact), int(y_exact)

        canvas = Image.new('RGB', (256 * 3, 256 * 3))
        valid_tiles = 0

        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:

                x_tile = center_x_tile + dx
                y_tile = center_y_tile + dy

                # Skip invalid tile indices
                if x_tile < 0 or y_tile < 0 or x_tile >= n or y_tile >= n:
                    continue

                url = url_template.format(z=zoom, x=x_tile, y=y_tile)

                try:
                    resp = requests.get(url, headers=headers, timeout=timeout)

                    if resp.status_code != 200:
                        continue

                    tile = Image.open(io.BytesIO(resp.content)).convert("RGB")

                    if is_valid_tile(tile):
                        canvas.paste(tile, ((dx + 1) * 256, (dy + 1) * 256))
                        valid_tiles += 1

                    # OPTIONAL DEBUG
                    # else:
                    #     tile.save(f"rejected_z{zoom}_{dx}_{dy}.png")

                except Exception:
                    continue

        # ✅ Accept zoom if enough valid tiles
        if valid_tiles >= min_valid_tiles:
            if verbose:
                print(f"[INFO] Using zoom {zoom} ({valid_tiles}/9 valid tiles)")

            offset_x_px = int((x_exact - center_x_tile) * 256)
            offset_y_px = int((y_exact - center_y_tile) * 256)

            left = 256 + offset_x_px - size[0] // 2
            top = 256 + offset_y_px - size[1] // 2

            return canvas.crop((left, top, left + size[0], top + size[1])), zoom

        # ❌ fallback to lower zoom
        zoom -= 1
        if verbose:
            print(f"[WARN] Falling back to zoom {zoom}")

    raise ValueError("No valid imagery found at any zoom level")

### Image processing

In [ ]:
RUN=False
# ==============================================================================
# PHASE 1: DYNAMIC SATELLITE & ROAD MAPS (TWO SEPARATE IMAGES)
# ==============================================================================
print("--- PHASE 1: GENERATING DYNAMIC SATELLITE & ROAD MAPS ---")
# df['lat_dec'] = df['lat'].apply(dms_to_decimal)
# df['lon_dec'] = df['lon'].apply(dms_to_decimal)

df_valid = df[(df['max_crater_radius_m'] > 0) & (df['lat_dec'].notna())].copy()
# df_valid['image_path_sat'] = ""
# df_valid['image_path_map'] = ""

for index, row in tqdm(df_valid.iterrows(), total=len(df_valid)):
    if(index!=423):
      continue
    else:
      print('ok')
    ari_path = os.path.join(output_dir, f"impact_{index}_ari.jpg")
    sat_path = os.path.join(output_dir, f"impact_{index}_sat.jpg")
    map_path = os.path.join(output_dir, f"impact_{index}_map.jpg")

    if not os.path.exists(ari_path) or not os.path.exists(sat_path) or not os.path.exists(map_path):
        try:
            opt_zoom = calculate_optimal_zoom(row['lat_dec'], row['max_crater_radius_m'], image_size=512)

            # 1. Fetch raw images
            # base_img_arieal = get_static_satellite_image(row['lat_dec'], row['lon_dec'], zoom=opt_zoom, size=(512, 512))
            base_img_sat, sat_zoom = get_esri_satellite_image(row['lat_dec'], row['lon_dec'], zoom=opt_zoom, size=(512, 512))
            # roadmap_img = get_static_roadmap_image(row['lat_dec'], row['lon_dec'], zoom=opt_zoom, size=(512, 512))


            # 2. Draw Blast Rings (Now with no fill and a thicker boundary)
            # final_img_ari = draw_bda_rings(base_img_arieal, row['lat_dec'], row['max_crater_radius_m'], zoom=opt_zoom)
            final_img_sat = draw_bda_rings(base_img_sat, row['lat_dec'], row['max_crater_radius_m'], zoom=sat_zoom)
            # final_floorplan_img = draw_bda_rings(roadmap_img, row['lat_dec'], row['max_crater_radius_m'], zoom=opt_zoom)

            # final_img_ari.save(ari_path)
            final_img_sat.save(sat_path)
            # final_floorplan_img.save(map_path)
        except Exception as e:
            print(f"Failed on index {index}: {e}")
            continue

    # df_valid.at[index, 'image_path_sat'] = sat_path
    # df_valid.at[index, 'image_path_map'] = map_path
    # df_valid.to_csv(csv_path, index=False)

--- PHASE 1: GENERATING DYNAMIC SATELLITE & ROAD MAPS ---


100%|██████████| 892/892 [00:00<00:00, 4360.87it/s]
